# Validation walkthrough: block → Gram Panchayat downscaling (Pune)

Reproducible notebook over the pipeline outputs (`python -m src.pipeline.run` first). Every number here
is read from `outputs/pune/reports/evaluation.json` or the held-out prediction files. Nothing is typed by hand.

In [ ]:
import json, sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.common.config import load_config
cfg = load_config(str(ROOT / "config/district_pune.yaml"))
ev = json.loads((cfg.paths.reports / "evaluation.json").read_text(encoding="utf-8"))
print(ev["headline"]["text"])

## 1. Validation design
Temporal hold-out (2026), blocked calibration weeks, 20 % of GPs never seen in training.

In [ ]:
pd.Series(ev["validation_design"])

## 2. Disaggregation skill (perfect prognosis: observed block value in)

In [ ]:
pp = ev.get("perfect_prognosis", {}).get("variables", {})
pd.DataFrame({v: {"RMSE block copy": d["overall"]["block_copy"]["rmse"],
                  "RMSE downscaler": d["overall"]["downscaler"]["rmse"],
                  "skill": d["overall"]["downscaler"]["skill_vs_block_copy"],
                  "skill (mass-conserving)": d["overall"]["downscaler_mass_conserving"]["skill_vs_block_copy"],
                  "within-block anomaly r": d["within_block_anomaly_correlation"]} for v, d in pp.items()}).T.round(3)

## 3. Operational forecast mode (ECMWF block forecast, lead 1-5): all methods, unseen GPs

In [ ]:
for v, d in ev["variables"].items():
    t = pd.DataFrame(d["unseen_gp"]["overall"]).T[["rmse", "mae", "bias", "r", "skill_vs_block_copy"]]
    print(v); display(t.round(3))

## 4. Skill by lead day (rain)

In [ ]:
bl = ev["variables"]["rain"]["unseen_gp"]["by_lead"]
lead = pd.DataFrame({k: {m: x["rmse"] for m, x in d.items()} for k, d in bl.items()}).T
ax = lead[["bl_block_copy", "bl_block_bias_corrected", "pred"]].plot(marker="o", figsize=(6, 3.5))
ax.set_xlabel("lead day"); ax.set_ylabel("RMSE (mm)"); ax.set_title("Rain RMSE by lead (unseen GPs)"); plt.show()

## 5. Rain detection at IMD thresholds

In [ ]:
cat = ev["variables"]["rain"]["categorical"]["unseen_gp"]
pd.DataFrame({(t, m): {k: x[k] for k in ("pod", "far", "csi", "ets", "freq_bias")}
              for t, mm in cat.items() for m, x in mm.items() if m in ("pred", "bl_block_copy")}).T.round(3)

## 6. Leave-one-block-out CV and the independent IMD gauge check

In [ ]:
display(pd.DataFrame(ev.get("leave_one_block_out", {})).T.drop(columns="folds", errors="ignore"))
ev.get("imd_independent_check")

## 7. Ground-truth QA: how well do the label sources agree with IMD gauges?

In [ ]:
ev.get("truth_qa")

## 8. One live inference with explanation
Re-run the operational model for the latest archived issue and look at one GP.

In [ ]:
from datetime import date
from src.pipeline.predict import run as predict
fc = pd.read_parquet(cfg.paths.interim / "block_forecasts.parquet")
issue = fc["issue_date"].max() - pd.Timedelta(days=5)
res = predict(cfg, issue.date(), "archive", write=False, languages=["en"])
g = res["gp_forecast"]; code_ = g.loc[g["rain_pred"].idxmax(), "gp_code"]
display(g[g["gp_code"] == code_][["lead_day", "fc_rain", "rain_pred", "rain_q10", "rain_q90", "rain_p_ge_2p5"]].round(2))
display(g[g["gp_code"] == code_].filter(like="shap_rain_").round(2))
print(res["bulletins"]["en"][code_]["sms"])